# EJEMPLO 2: PRUEBA Z DE UNA MUESTRA (BILATERAL)

Supongamos que somos científicos de datos y desarrollamos un modelo de Machine Learning para apoyo diagnóstico de enfermedades cardiacas. El modelo tiene un desempeño promedio (F1-score) de 0.86 y una desviación estándar de 0.03 y estos parámetros se han obtenido tras aplicar pruebas durante los últimos dos años en varios miles de pacientes.

**Ahora hemos hecho modificaciones al modelo y queremos asegurarnos de que su desempeño no ha cambiado**

## 1. Desarrollo de la prueba de hipótesis (z-test de una muestra bilateral)

En este caso se trata de una prueba bilateral porque el nuevo modelo podría generar un **incremento o reducción** del desempeño, pero queremos asegurarnos que desde el punto de vista estadístico en esencia el desempeño es el mismo del modelo original.

### 1.1. Paso 1: definir el problema del negocio

> **Hemos hecho modificaciones al modelo y queremos asegurarnos de que su desempeño no ha cambiado**

### 1.2. Paso 2: redactar el problema del negocio como un problema de Ciencia de Datos/Machine Learning

> ¿El nuevo modelo tiene un desempeño equivalente, desde el punto de vista estadístico, al modelo original?**

### 1.3. Paso 3: definir $H_0$ y $H_1$

- $H_0$: el promedio de desempeño del nuevo modelo es igual al del modelo anterior $\rightarrow \bar{x} = \mu$
- $H_1$: el promedio de desempeño del nuevo modelo es **diferente** que el del modelo anterior $\rightarrow \bar{x} \neq \mu$ 

*Nota: recordemos que $\mu = 0.86$ y $\sigma = 0.03$*

### 2.4. Paso 4: definir $\alpha$

Asumiremos un nivel de significancia $\alpha = 0.05$. Podemos dibujar este nivel de significancia en la distribución Z usando alguna [herramienta online](https://www.infrrr.com/distributions/normal-distributions):

![](distribucion_z_test_bilateral.png)

En este gráfico:
- El eje horizontal es la variable $z$
- Los valores de $z$ correspondientes a un $\alpha = 0.05/2$ son $\pm 1.96$
- Las regiones azules corresponden a las **colas derecha e izquierda** (prueba bilateral porque puede haber incremento o reducción del desempeño) y contienen el 5% de la distribución
- Si tras realizar la prueba el valor de $z$ calculado está dentro cualquiera de las zonas azules, nos inclinaremos por $H_1$ y rechazaremos $H_0$. De lo contrario nos inclinamos por $H_0$ y rechazamos $H_1$

### 1.5. Definir la potencia de la prueba ($1-\beta$) y el tamaño de la muestra ($n$)

Supongamos que esperamos una reducción máxima de 0.01 (0.86 a 0.85). En el caso de la prueba z el tamaño del efecto será:

In [19]:
# Tamaño del efecto (d)
x_esperado = 0.85
mu = 0.86
sigma = 0.03
d = (x_esperado-mu)/sigma
print(f"Tamaño del efecto: {d}")

Tamaño del efecto: -0.33333333333333365


El signo menos simplemente indica que esperamos que haya una ligera reducción. Lo que nos interesa es la magnitud (0.333) lo que nos dice que esperamos un efecto "pequeño" (que es lo que buscamos, pues esperamos que el nuevo modelo tenga un desempeño similar al modelo antiguo).

Supongamos ahora que queremos una potencia de la prueba de 0.9. Con esta información ya podemos calcular el tamaño de la muestra:

In [20]:
from statsmodels.stats.power import NormalIndPower

# Definir parámetros de nuestra prueba para el cálculo de n
effect_size = abs(d) # Tomaremos el valor absoluto del tamaño del efecto calculado anteriormente
power = 0.9
alpha = 0.05

# Instancia de NormalIndPower
analisis = NormalIndPower()

# Y cálculo del tamaño de la muestra
n = analisis.solve_power(
    effect_size=abs(d), # Tomaremos el valor absoluto del tamaño del efecto calculado anteriormente
    alpha = alpha,
    power=power,
    alternative='two-sided', # larger = unilateral derecho, smaller = unilateral izquierdo, two-sided: bilateral
    ratio = 0 # No estamos calculando proporción entre medias
)
print(f"Tamaño sugerido de la muestra: {n}")

Tamaño sugerido de la muestra: 94.56677578333222


Así que requerimos un tamaño de muestra de al menos 95. En este caso cada muestra será una prueba del desempeño del nuevo modelo con un set de datos diferente.

### 1.5. Paso 5: recolectar y preparar los datos

Supondremos que hemos recolectado $n=100$ datos de desempeño del nuevo modelo. A continuación podemos ver estos desempeños:

In [21]:
import pandas as pd

df = pd.read_csv('dataset_ztest_bilateral.csv')
df

,f1_score
0,0.812431
1,0.874920
2,0.853489
3,0.799811
4,0.827642
...,...
95,0.875933
96,0.812463
97,0.804096
98,0.856382


Y podemos verificar que tenemos 100 mediciones (100 datos). Veamos el promedio de estos desempeños:

In [22]:
df.mean()

f1_score    0.845813
dtype: float64

El desempeño promedio de este nuevo modelo es de $\bar{x}=0.845$ que es muy similar al desempeño del sistema original ($\mu=0.86$).

Y recordemos que lo que nos interesa es **determinar si el desempeño del nuevo modelo es estadísticamente diferente del modelo original**

Aprovechemos para verificar la normalidad de esta distribución:

In [23]:
from scipy.stats import shapiro

W, p_shapiro = shapiro(df['f1_score'])
p_shapiro

0.2748704400205805

Como p>0.05 no rechazamos la hipótesis nula y por tanto los tienen una distribución normal.

### 1.6. Paso 6: aplicar la prueba estadística para obtener el valor p



Hagamos el análisis manual y luego veremos cómo hacerlo usando Scipy.

Comencemos calculando $z$:

$$z = \frac{\bar{x}-\mu}{\sigma/\sqrt{n}}=\frac{0.845-0.86}{0.03/\sqrt{100}}=-5$$

A continuación verificamos dónde se encuentra ubicado este valor de $z$ dentro de la distribución:

![](distribucion_z_test_bilateral.png)

Y con esto verificamos que $z$ está en la zona sombreada del lado izquierdo y esto nos indica que podemos rechazar $H_0$ e inclinarnos por $H_1$.

Veamos como llegar a este mismo resultado usando Scipy:


In [24]:
from scipy.stats import norm
import numpy as np

# Parámetros de la población y de la muestra
mu = 0.86
sigma = 0.03
n = len(df)
x_barra = np.mean(df)

# Calcular z
z = (x_barra-mu)/(sigma/np.sqrt(n))

# Calcular p = 2 * P(Z<=z) (con z siendo negativo)
p = 2*norm.cdf(z)

print(f'z: {z}')
print(f'p: {p}')

z: -4.72890926509643
p: 2.257292273329003e-06


### 1.7. Paso 7: aceptar o rechazar $H_0$

Y vemos que $p = 2.25x10{-6} <0.05$ y por tanto rechazamos la hipótesis nula (medias iguales) y nos inclinamos por $H_1$ (medias diferentes).

Y por tanto:

> El nuevo modelo no tiene el mismo desempeño del modelo anterior y de hecho hay una reducción en el desempeño (0.86 vs. 0.845)

### 1.8. Paso 8: evaluar el tamaño del efecto actualizado

Hemos visto que hay un efecto pero ¿qué tan grande es?

Simplemente recalculamos el tamaño del efecto con la media de los datos obtenidos:

In [25]:
# Tamaño del efecto actualizado
d = (np.mean(df)-mu)/sigma
print(f'Tamaño del efecto (d) actualizado: {d}')

Tamaño del efecto (d) actualizado: -0.472890926509643


Esto quiere decir que:

> El nuevo desempeño se reduce 0.47 desviaciones estándar del valor original ❌❌❌

Además, teniendo en cuenta que no usamos el mínimo de datos (94), sino un poco más (100), verifiquemos la potencia final de esta prueba:

In [26]:
# Definir parámetros de la prueba actualizada (con el nuevo n)
n = len(df)
alpha = 0.05 # Nivel de significancia

# Instancia de NormalIndPower
analisis = NormalIndPower()

# Potencia actualizada
potencia = analisis.power(
    effect_size=d, # El valor calculado en el bloque de código anterior
    nobs1 = n, # tamaño de la muestra usada
    alpha = alpha,
    ratio = 0, # porque es prueba de 1 muestra
    alternative = 'two-sided' # porque es unilateral derecha
)
print(f'Potencia actualizada de la prueba: {potencia}')

Potencia actualizada de la prueba: 0.9971880960366131


Lo anterior quiere decir que:

> Si la reducción del desempeño realmente existe hay un 99.7% de probabilidad de que esta prueba lo detecte ✅✅✅

**Es decir que con esto concluimos que el nuevo modelo empeora el desempeño de las predicciones en comparación con el modelo anterior👎👎👎**